# Phase 2 — SFT LoRA puis DPO de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA, puis alignement par préférences (DPO) du modèle SFT.

- **Prérequis** : GPU, fichiers `data/v4/anonymized/sft_{mediqal_oeq,mediqal_mcqu,mediqal_mcqm,ultramedical}_{train,validation,test}.anonymized.jsonl` produits par `prep_SFT_MediQAl_oeq`, `prep_SFT_MediQAl_mcqu`, `prep_SFT_MediQAl_mcqm` (français) et `prep_SFT_UltraMedical` (anglais), sélectionnés pour le triage (environ 3 000 paires en français, 2 000 en anglais). **Une ligne = un cas clinique** : le cas, puis ses questions dans l'ordre, chacune suivie de sa réponse (questions ouvertes, ou QCM avec propositions A à E et consigne). La perte porte sur **toutes les réponses de l'assistant** (`assistant_only_loss`), chacune avec l'historique qui la précède.
- **Prérequis DPO** : fichiers `data/v4/anonymized/dpo_{mediqal,ultramedical}_{train,validation,test}.anonymized.jsonl` produits par `prep_DPO_MediQAl` (français : QCM, `rejected` = corrigé avec une erreur construite) et `prep_DPO_UltraMedical` (anglais : paires vérifiées par le corrigé ou par un écart de note net, test revu par des experts).
- **Étapes** : données → modèle → LoRA → entraînement SFT → validation intermédiaire → DPO → validation DPO → modèle final fusionné.
- **Suivi** : MLflow sur DagsHub (`MLFLOW_BACKEND = "dagshub"`), consultable de partout et quelle que soit la machine d'entraînement. En local (`"local"`) : `mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet.

In [31]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow dagshub pandas

## 1. Configuration

In [32]:
import functools
import gc
import json
import math
import re
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import DatasetDict, concatenate_datasets, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer
from trl.chat_template_utils import is_chat_template_stop_token_trained

from projet_14.mediqal import LETTERS, answer_letters

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Données SFT de la phase 1, anonymisées et sélectionnées pour le triage (~3 000 paires FR, 2 000 EN) : fichier -> langue.
# Une ligne = un cas clinique (conversation multi-tours) ; en anglais, une question
SFT_DATA_DIR = Path("../data/v4/anonymized")
SFT_DATASETS = {
    "sft_mediqal_oeq": "fr",    # questions ouvertes, réponses rédigées, avec cas clinique
    "sft_mediqal_mcqu": "fr",   # QCM à réponse unique, avec cas clinique
    "sft_mediqal_mcqm": "fr",   # QCM à réponses multiples (liste), avec cas clinique
    "sft_ultramedical": "en",   # vignettes cliniques (MedQA…) et patients (ChatDoctor), réponses rédigées en anglais
}

# Suivi MLflow : "dagshub" (serveur distant) ou "local" (fichier ../mlflow.db)
MLFLOW_BACKEND = "dagshub"
DAGSHUB_REPO_OWNER = "obiyohan"
DAGSHUB_REPO_NAME = "my-ml-mlflow"

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = True  # True : run rapide (quelques lignes par source, dont les plus longues), False : run complet

SEED = 42
# Version des données de la phase 1 : dans le nom des runs, pour ne pas écraser les checkpoints d'une version à l'autre
# v1 : propositions absentes du prompt ; v2 : propositions dans le prompt, répartition par cas clinique,
#      DPO anglais équilibré sur la longueur ; v3 : consigne « une seule / plusieurs propositions exactes » ;
# v4 : une ligne par cas clinique (questions dans l'ordre du récit), questions ouvertes ajoutées, perte sur toutes les réponses
DATA_VERSION = "v4"
RUN_NAME = f"sft-lora-r16-{DATA_VERSION}-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

device=cuda, dtype=torch.bfloat16


## 2. Données : train / validation / test

In [33]:
# Format conversationnel (colonne `messages`) : une ligne = un cas clinique et toutes ses questions.
# Avec assistant_only_loss (section 5), la perte porte sur toutes les réponses de l'assistant, pas sur le cas ni les questions.
# Chaque source garde son découpage train / validation / test (tiré du cas clinique) ; les sources sont ensuite mélangées
parts = {split: [] for split in ("train", "validation", "test")}
for stem, lang in SFT_DATASETS.items():
    files = {split: str(SFT_DATA_DIR / f"{stem}_{split}.anonymized.jsonl") for split in parts}
    ds = load_dataset("json", data_files=files).select_columns(["messages", "n_questions"])
    for split, part in ds.items():
        parts[split].append(part.add_column("source", [stem] * len(part)).add_column("lang", [lang] * len(part)))

splits = DatasetDict({split: concatenate_datasets(p).shuffle(seed=SEED) for split, p in parts.items()})

# Petit run : quelques lignes de chaque source, pour que chaque format et chaque évaluation soient exercés,
# plus les lignes les plus longues du train, pour tester la mémoire GPU dans le pire cas
if SMALL_RUN:
    SMALL_SIZES = {"train": 12, "validation": 4, "test": 6}  # lignes par source
    SMALL_LONGEST = 4                                        # + lignes les plus longues du train
    for name, size in SMALL_SIZES.items():
        ds = splits[name]
        keep = [i for stem in SFT_DATASETS for i in [j for j, s in enumerate(ds["source"]) if s == stem][:size]]
        if name == "train":
            counter = AutoTokenizer.from_pretrained(MODEL_NAME)  # longueur en tokens (en caractères, l'anglais serait favorisé)
            length = [len(counter("\n".join(m["content"] for m in messages))["input_ids"]) for messages in ds["messages"]]
            keep += sorted(range(len(ds)), key=lambda i: length[i], reverse=True)[:SMALL_LONGEST]
        splits[name] = ds.select(sorted(set(keep)))

# Lignes (cas) et questions (réponses apprises) par source et par split
pd.concat({
    "lignes": pd.DataFrame({name: pd.Series(ds["source"]).value_counts() for name, ds in splits.items()}),
    "questions": pd.DataFrame({name: pd.DataFrame({"source": ds["source"], "n": ds["n_questions"]}).groupby("source")["n"].sum()
                               for name, ds in splits.items()}),
}, axis=1).fillna(0).astype(int)


lignes                 questions                
                  train validation test     train validation test
sft_mediqal_mcqm     12          4    6        40         13   16
sft_mediqal_mcqu     15          4    6        53         17   26
sft_mediqal_oeq      13          4    6        43         13   17
sft_ultramedical     12          4    6        12          4    6

## 3. Tokenizer et modèle

In [34]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML, avec une réponse de l'assistant terminée par <|endoftext|> (eos_token) au lieu de <|im_end|>.
# Dans Qwen3-Base, <|im_start|> et <|im_end|> n'ont jamais été entraînés (embeddings à leur valeur d'initialisation)
# et LoRA ne modifie pas les embeddings : le modèle ne peut pas apprendre à produire <|im_end|>, donc à s'arrêter.
# <|endoftext|> a été appris au pré-entraînement : le modèle sait le prédire, et vLLM s'arrête dessus par défaut.
# {% generation %} ... {% endgeneration %} : délimite les tokens appris (assistant_only_loss). Ils n'apparaissent pas dans
# le texte rendu : le prompt vu par vLLM est identique. <|endoftext|> est à l'intérieur : le modèle apprend à s'arrêter.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' }}"
    "{% if message['role'] == 'assistant' %}"
    "{% generation %}{{ message['content'] + eos_token }}{% endgeneration %}"
    "{% else %}{{ message['content'] + '<|im_end|>\n' }}{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)
assert is_chat_template_stop_token_trained(tokenizer), "<|endoftext|> doit être dans le masque des réponses"

# Une ligne à plusieurs questions, et les tokens appris (masque des réponses de l'assistant)
example = next(e for e in splits["train"] if e["n_questions"] >= 3)
print(tokenizer.apply_chat_template(example["messages"], tokenize=False))
encoded = tokenizer.apply_chat_template(example["messages"], tokenize=True, return_dict=True, return_assistant_tokens_mask=True)
learned = [token for token, mask in zip(encoded["input_ids"], encoded["assistant_masks"]) if mask]
print(f"=== {len(learned)} tokens appris sur {len(encoded['input_ids'])} :\n{tokenizer.decode(learned)}")


<|im_start|>system
Tu es un assistant médical expert. Réponds de façon précise et concise.<|im_end|>
<|im_start|>user
Cas clinique : Une femme de 55 ans vient consulter pour un ictère apparu progressivement, sans fièvre, ni douleur abdominale. Dans ses antécédents, on note une hypertension artérielle traitée depuis deux ans par de l'Aldomet® (alphaméthyl-dopa), et pour une arthrose cervico-dorsale du Dupéran (clométacine). Elle est en outre constipée chronique et prend depuis plusieurs années des laxatifs (Bisacodyl, Séné, Dioctylsulfosuccinate). Elle consomme en moyenne 20 g d'alcool par jour. A l'examen clinique, on note une ascite modérée, des oedèmes des membres inférieurs ; elle se plaint d'une asthénie avec somnolence diurne et parfois désorientation. Biologiquement, il existe un bloc bêta-gamma à l'électrophorèse des protides, une vitesse de sédimentation à 35 mm à la première heure, des triglycérides à 1, 2 mmol/l ; un fer sérique normal, une ferritinémie à une fois et demie la

In [35]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:02<00:00, 107.24it/s]


1.72 B paramètres


## 4. LoRA

In [36]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [37]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    # Jeton lu dans la variable d'environnement DAGSHUB_USER_TOKEN,
    # sinon connexion via le navigateur au premier lancement (jeton mémorisé ensuite)
    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"Suivi MLflow : {mlflow.get_tracking_uri()}")

mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "data_version": DATA_VERSION, "assistant_only_loss": True, "small_run": SMALL_RUN, "device": device, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    # Une époque : au run v2 (2 époques), eval_loss était au plus bas en fin d'époque 1 puis remontait légèrement
    num_train_epochs=1,
    max_steps=30 if SMALL_RUN else -1,
    # v4 : lignes plus longues (un cas entier, jusqu'à ~1 460 tokens) : 2 lignes par lot, même batch effectif
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # batch effectif = 16 lignes (cas cliniques)
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1536,  # aucune ligne v4 n'est coupée (max ~1 460 tokens) : toutes les réponses sont apprises
    assistant_only_loss=True,  # perte sur toutes les réponses de l'assistant, pas sur le cas ni les questions
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    # v4 : ~2 100 lignes en train, soit ~130 étapes par époque : une évaluation et un checkpoint toutes les 25 étapes
    eval_steps=10 if SMALL_RUN else 25,
    save_steps=10 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Suivi MLflow : https://dagshub.com/obiyohan/my-ml-mlflow.mlflow


Truncating train dataset: 100%|██████████| 52/52 [00:00<00:00, 3412.51 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 52/52 [00:00<00:00, 7865.27 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 16/16 [00:00<00:00, 3473.54 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [38]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

Training Loss,Validation Loss,Step,Before Loss,Entropy,Mean Token Accuracy,Num Tokens
No log,No log,0,1.375039,1.414849,0.690063,0.000000


{'val_before_loss': 1.3750392198562622,
 'eval_entropy': 1.414848506450653,
 'eval_mean_token_accuracy': 0.6900632679462433,
 'eval_num_tokens': 0.0}

In [39]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.282429,1.273032,1.228426,0.705082,111050.000000
20,1.058230,1.303936,1.073425,0.707016,211775.000000
30,0.858728,1.317399,1.047113,0.705393,322741.000000


{'train_runtime': 195.9736,
 'train_samples_per_second': 2.449,
 'train_steps_per_second': 0.153,
 'total_flos': 3425124304171008.0,
 'train_loss': 1.0840366999308269,
 'epoch': 7.615384615384615}

In [40]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data_version": DATA_VERSION,
    "data": {name: {"lignes": len(ds), "questions": sum(ds["n_questions"])} for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

4747

## 6. Validation intermédiaire

1. **Loss, perplexité et token accuracy** sur validation (avant / après SFT) et test.
2. **Qualité des réponses générées** sur des questions de test (QCM `mcqu` et `mcqm` ; chaque question d'un cas avec l'historique de référence qui la précède), modèle de base vs modèle SFT, sur les **lettres** données dans la réponse. Génération par lots pour bien charger le GPU.
   - `accuracy` : exactement les lettres attendues ;
   - `option_accuracy` : exactitude par proposition, part des cinq propositions correctement cochées ou laissées. Utile en `mcqm` ; en `mcqu`, une réponse vide obtient déjà 0,8 : y lire l'accuracy ;
   - `f1` : crédit partiel pour les réponses multiples. Il favorise les réponses qui cochent trop : en `mcqm`, tout cocher donne un F1 d'environ 0,7 ;
   - `extra_letters` : lettres données en trop (positif) ou en moins (négatif) par rapport au corrigé, en moyenne.

   Deux **réponses naïves**, identiques pour toutes les questions, servent de référence : `frequente` (la réponse la plus fréquente du train) et `tout_cocher` (A à E). Un modèle utile doit les battre, sur l'accuracy et l'exactitude par proposition.

La **latence** n'est pas mesurée ici : avec l'adaptateur LoRA non fusionné et Transformers, elle ne serait pas représentative. Elle sera mesurée en phase 3, sur le modèle final fusionné servi par vLLM.

In [41]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

# token_accuracy : part des tokens de la réponse attendue que le modèle prédit correctement
# (TRL la nomme « eval_mean_token_accuracy » quel que soit le préfixe : on la renomme ici)
evaluations = {"val_before": val_before, "val_after": val_after, "test": test_metrics}
summary = pd.DataFrame({
    name: {"loss": m[f"{name}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
    for name, m in evaluations.items()
}).T
summary["perplexity"] = summary["loss"].map(math.exp)

# Même mesure sur le test, source par source : chaque format doit être appris (oeq : réponses rédigées, évaluées ici seulement)
per_source = {}
for source in SFT_DATASETS:
    subset = splits["test"].filter(lambda example: example["source"] == source)
    if len(subset):
        m = trainer.evaluate(subset, metric_key_prefix=f"test_{source}")
        per_source[f"test_{source}"] = {"loss": m[f"test_{source}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
per_source_summary = pd.DataFrame(per_source).T
per_source_summary["perplexity"] = per_source_summary["loss"].map(math.exp)
pd.concat([summary, per_source_summary])

Training Loss,Validation Loss,Step,After Loss,Entropy,Mean Token Accuracy,Num Tokens
0.858728,No log,30,1.273032,1.228426,0.705082,322741.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 24/24 [00:00<00:00, 4048.07 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.858728,1.157444,30,1.075683,0.742296,322741.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 6/6 [00:00<00:00, 1635.21 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Oeq Loss,Entropy,Mean Token Accuracy,Num Tokens
0.858728,No log,30,2.316328,2.143785,0.543867,322741.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 6/6 [00:00<00:00, 1623.92 examples/s]


... (2 outputs left out)

Training Loss,Validation Loss,Step,Sft Mediqal Mcqm Loss,Entropy,Mean Token Accuracy,Num Tokens
0.858728,No log,30,0.134844,0.148696,0.944113,322741.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 6/6 [00:00<00:00, 1243.25 examples/s]


Training Loss,Validation Loss,Step,Sft Ultramedical Loss,Entropy,Mean Token Accuracy,Num Tokens
0.858728,No log,30,0.779909,0.733020,0.779106,322741.000000


,loss,token_accuracy,perplexity
val_before,1.375039,0.690063,3.955232
val_after,1.273032,0.705082,3.571667
test,1.157444,0.742296,3.181791
test_sft_mediqal_oeq,2.316328,0.543867,10.138378
test_sft_mediqal_mcqu,0.117441,0.955363,1.124615
test_sft_mediqal_mcqm,0.134844,0.944113,1.144359
test_sft_ultramedical,0.779909,0.779106,2.181273


In [42]:
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [IM_END_ID, tokenizer.eos_token_id]
# Réponses mcqu : « C. texte de la proposition ». La lettre vient en premier : une réponse coupée reste évaluable.
# Le modèle de base, qui ne s'arrête pas toujours, est ainsi coupé plus tôt.
MAX_NEW_TOKENS = 64


def render(prompt):
    return tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate_batch(prompts, batch_size=16, max_new_tokens=MAX_NEW_TOKENS, desc=None, model=None):
    # Génération par lots (GPU bien chargé) : renvoie les réponses, dans l'ordre des prompts
    model = model or trainer.model  # par défaut, le modèle du SFT
    texts = [render(prompt) for prompt in prompts]
    # Prompts triés par longueur : moins de padding dans chaque lot, les plus longs d'abord
    # (un manque de mémoire GPU apparaît dès le premier lot)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    answers = [None] * len(texts)
    padding_side, tokenizer.padding_side = tokenizer.padding_side, "left"  # padding à gauche : la génération suit chaque prompt
    try:
        for start in tqdm(range(0, len(order), batch_size), desc=desc, unit="lot"):
            batch = order[start:start + batch_size]
            inputs = tokenizer([texts[i] for i in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id,
            )
            decoded = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, answer in zip(batch, decoded):
                answers[i] = answer.strip()
    finally:
        tokenizer.padding_side = padding_side
    return answers


def question_examples(dataset):
    # Une ligne (cas) -> une question par réponse de l'assistant : prompt = conversation jusqu'à elle, référence = sa réponse
    prompts, references = [], []
    for messages in dataset["messages"]:
        for k, message in enumerate(messages):
            if message["role"] == "assistant":
                prompts.append(messages[:k])
                references.append(message["content"])
    return prompts, references


QUALITY_METRICS = ["accuracy", "option_accuracy", "f1", "extra_letters"]
NAIVE_ANSWERS = ["frequente", "tout_cocher"]


def letter_scores(predicted: frozenset, expected: frozenset) -> dict:
    # Mesures sur les lettres d'une réponse (ensembles de lettres A à E)
    common = len(predicted & expected)
    return {
        "accuracy": float(predicted == expected),  # exactement les lettres attendues
        # Exactitude par proposition : part des 5 propositions correctement cochées ou laissées
        "option_accuracy": sum((letter in predicted) == (letter in expected) for letter in LETTERS) / len(LETTERS),
        # F1 : crédit partiel ; favorise les réponses qui cochent trop (tout cocher : ~0,7 en mcqm)
        "f1": 2 * common / (len(predicted) + len(expected)) if common else 0.0,
        "extra_letters": len(predicted) - len(expected),  # > 0 : coche trop ; < 0 : pas assez
    }


def letters_text(letters) -> str:
    # Réponse réduite à ses lettres (« A.\nC. »), lisible par answer_letters
    return "\n".join(f"{letter}." for letter in sorted(letters))


def naive_answers(train_answers: list[str]) -> dict:
    # Réponses naïves, identiques pour toutes les questions : un modèle utile doit les battre
    most_common = Counter(answer_letters(answer) for answer in train_answers).most_common(1)[0][0]
    return {"frequente": letters_text(most_common), "tout_cocher": letters_text(LETTERS)}


def add_scores(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Ajoute <mesure>_<colonne> pour chaque colonne de réponses, comparée à df["reference"]
    expected = [answer_letters(reference) for reference in df["reference"]]
    for column in columns:
        scores = pd.DataFrame(
            [letter_scores(answer_letters(answer), e) for answer, e in zip(df[column], expected)], index=df.index
        )
        for metric in QUALITY_METRICS:
            df[f"{metric}_{column}"] = scores[metric]
    return df


def score_table(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Moyennes : une ligne par mesure, une colonne par modèle ou réponse naïve
    return pd.DataFrame({column: {metric: df[f"{metric}_{column}"].mean() for metric in QUALITY_METRICS} for column in columns})

In [43]:
N_EVAL = 20 if SMALL_RUN else 200    # questions de test par source
BATCH_SIZE = 16                      # à baisser en cas de manque de mémoire GPU (OutOfMemoryError)

# Sources évaluées par génération, avec leur nombre maximal de tokens générés.
# Les réponses rédigées (oeq, anglais) ne se comparent pas mot à mot à une référence :
# elles sont évaluées par la loss et la perplexité (cellule précédente).
GENERATION_SOURCES = {"sft_mediqal_mcqu": MAX_NEW_TOKENS, "sft_mediqal_mcqm": 192}
ANSWER_COLUMNS = ["base", "sft", *NAIVE_ANSWERS]  # modèles, puis réponses naïves de référence

trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération

# Génération par lots, modèle de base puis modèle SFT, source par source
source_generations = []
for source, max_new_tokens in GENERATION_SOURCES.items():
    # Chaque question d'un cas, avec l'historique de référence (questions et bonnes réponses précédentes)
    prompts, references = question_examples(splits["test"].filter(lambda example: example["source"] == source))
    prompts, references = prompts[:N_EVAL], references[:N_EVAL]
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"Base {source}")
    sft_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"SFT {source}")
    train_answers = question_examples(splits["train"].filter(lambda example: example["source"] == source))[1]
    source_generations.append(pd.DataFrame({
        "source": source,
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": references,
        "base": base_answers,
        "sft": sft_answers,
        **naive_answers(train_answers),  # mêmes réponses pour toutes les questions de la source
    }))
generations = add_scores(pd.concat(source_generations, ignore_index=True), ANSWER_COLUMNS)
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)

# Moyennes sur toutes les sources, puis source par source
comparison = score_table(generations, ANSWER_COLUMNS)
quality_by_source = pd.concat({source: score_table(group, ANSWER_COLUMNS) for source, group in generations.groupby("source")})
display(quality_by_source)
comparison

Filter: 100%|██████████| 52/52 [00:00<00:00, 9268.79 examples/s]


base       sft  frequente  tout_cocher
sft_mediqal_mcqm accuracy         0.18750  0.000000   0.125000     0.125000
                 option_accuracy  0.58750  0.587500   0.537500     0.512500
                 f1               0.53125  0.606944   0.483929     0.658482
                 extra_letters    1.06250  0.687500  -0.562500     2.437500
sft_mediqal_mcqu accuracy         0.20000  0.400000   0.250000     0.000000
                 option_accuracy  0.57000  0.760000   0.700000     0.200000
                 f1               0.30000  0.400000   0.250000     0.333333
                 extra_letters    1.05000  0.000000   0.000000     4.000000

,base,sft,frequente,tout_cocher
accuracy,0.194444,0.222222,0.194444,0.055556
option_accuracy,0.577778,0.683333,0.627778,0.338889
f1,0.402778,0.491975,0.353968,0.477844
extra_letters,1.055556,0.305556,-0.250000,3.305556


In [44]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations.groupby("source")[["source", "question", "reference", "base", "sft"]].head(3)

,source,question,reference,base,sft
0,sft_mediqal_mcqu,ale\nC. L'adaptation du traitement de l'hypertension artérielle\nD. La mise en place d'une perfusion de mannitol a 20%\nE. La protection des voies aériennes supérieures\n\nUne seule proposition es...,D. La mise en place d'une perfusion de mannitol a 20%,La réponse fausse est : **A. L'administration de diazépam par voie générale**.\n\nExplication : \n- **Diazépam** est un anxiolytique et n'est pas utilisé pour traiter l'éclampsie. \n- **Sulf,A. L'administration de diazépam par voie générale
1,sft_mediqal_mcqu,Une réanimation métabolique seule\nD. Une intubation associée à une ventilation et une réanimation métabolique\nE. Une intubation associée à une ventilation uniquement\n\nUne seule proposition est...,A. Une désobstruction associée à de l'oxygène nasal,D. Une intubation associée à une ventilation et une réanimation métabolique\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\n.waitKey\...,A. Une désobstruction associée à de l'oxygène nasal
2,sft_mediqal_mcqu,Le DIU (dispositif intra-utérin)\nB. La pilule micro-progestative en continu\nC. Les macro-progestatifs\nD. La pilule minidosée\nE. Les propositions A et C sont exactes\n\nUne seule proposition e...,A. Le DIU (dispositif intra-utérin),A. Le DIU (dispositif intra-utérin)\nB. La pilule micro-progestative en continu\nC. Les macro-progestatifs\nD. La pilule minidosée\nE. Les propositions A et C sont exactes\n\nUne seule proposition...,A. Le DIU (dispositif intra-utérin)
20,sft_mediqal_mcqm,. Ce taux est compatible avec le diagnostic de méningite virale\nE. Ce taux est compatible avec le diagnostic de méningite à Listeria monocytogénès\n\nPlusieurs propositions sont exactes (au moins...,- C. Ce taux évoque très fortement la présence d'une bactérie dans le LCR\n- E. Ce taux est compatible avec le diagnostic de méningite à Listeria monocytogénès,Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态：Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态：Tu es un assistant médical expert. R...,B. Il s'agit d'un taux incompatible avec le diagnostic de méningite tuberculeuse\nC. Ce taux évoque très fortement la présence d'une bactérie dans le LCR
21,sft_mediqal_mcqm,. Myélogramme\nC. Gastroscopie avec biopsie\nD. Dosages de vitamine B12 et de folates dans le sang\nE. Recherche d'anticorps anti-facteur intrinsèque\n\nPlusieurs propositions sont exactes (au moi...,- B. Myélogramme\n- D. Dosages de vitamine B12 et de folates dans le sang,Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n��态：Tu es un assistant médical expert. Réponds de façon précise et concise.��态\n状态：Tu es un assistant médical expert. Ré...,A. Tubage gastrique\nB. Myélogramme\nC. Gastroscopie avec biopsie\nD. Dosages de vitamine B12 et de folates dans le sang
22,sft_mediqal_mcqm,B. Trouble d'absorption de la vitamine B12\nC. Carence d'apport en folates\nD. Trouble d'absorption des folates\nE. Trouble d'utilisation des folates\n\nPlusieurs propositions sont exactes (au moi...,- C. Carence d'apport en folates\n- D. Trouble d'absorption des folates,- A. Carence d'apport en vitamine B12\n- C. Carence d'apport en folates\n- D. Trouble d'absorption des folates\n- E. Trouble d'utilisation des folates\n\nQuestion : Quel est le principal facteur c...,- A. Carence d'apport en vitamine B12\n- D. Trouble d'absorption des folates


In [45]:
# Métriques finales dans MLflow (les loss sont déjà envoyées par le Trainer), puis clôture du run
mlflow.log_metrics({f"{name}_token_accuracy": row["token_accuracy"] for name, row in summary.iterrows()})
mlflow.log_metrics({f"{name}_perplexity": row["perplexity"] for name, row in summary.iterrows()})
mlflow.log_metrics({
    f"test_{metric}_{model}": comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in comparison.columns
})
# Par source : loss, token accuracy et perplexité du test, mesures des réponses générées et des réponses naïves
mlflow.log_metrics({
    f"{name}_{metric}": value for name, row in per_source_summary.iterrows() for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_{source}_{metric}_{column}": value
    for (source, metric), row in quality_by_source.iterrows()
    for column, value in row.items()
})
mlflow.end_run()

🏃 View run sft-lora-r16-v4-seed42-small at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2/runs/b09ebe8a41d24f62b8e363b166c4c44e
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2


## Lecture des résultats

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | courbes de loss d'entraînement et de validation | baissent ensemble ; si `eval_loss` remonte : surapprentissage |
| `val_before_loss`, `val_after_loss`, `test_loss` | loss avant / après SFT et sur le test | après < avant ; test proche de val |
| `*_token_accuracy` | part des tokens de la réponse correctement prédits | augmente après SFT |
| `*_perplexity` | exp(loss) | baisse après SFT |
| `test_sft_<source>_loss`, `_token_accuracy`, `_perplexity` | les mêmes mesures sur le test, source par source | chaque source s'améliore, y compris les questions ouvertes (`oeq`) |
| `test_accuracy_*` | réponse générée avec exactement les lettres attendues (sources françaises) | sft > base, sft > `frequente` et `tout_cocher` |
| `test_option_accuracy_*` | exactitude par proposition : part des 5 propositions correctement cochées ou laissées | sft > `frequente` et `tout_cocher` en `mcqm` ; en `mcqu`, une réponse vide obtient 0,8 : lire l'accuracy |
| `test_f1_*` | F1 sur les lettres (crédit partiel pour les réponses multiples) | sft > base ; à lire avec prudence, `tout_cocher` obtient ~0,7 en `mcqm` |
| `test_extra_letters_*` | lettres en trop (> 0) ou en moins (< 0) par rapport au corrigé | proche de 0 |
| `test_sft_<source>_<mesure>_*` | les mêmes mesures, source par source | sft bat les réponses naïves pour `mcqu` et `mcqm` |

- La courbe `eval_mean_token_accuracy` mélange toutes les évaluations (TRL ignore le préfixe) : utiliser `*_token_accuracy`.
- Les mesures portent sur les lettres : une réponse sans lettre (format non appris) compte 0.
- `*_frequente`, `*_tout_cocher` : réponses naïves, identiques pour toutes les questions (réponse la plus fréquente du train, A à E). Elles ne dépendent pas du modèle et donnent le niveau à dépasser.
- La latence (TTFT, TPOT, débit) sera mesurée en phase 3, sur le modèle final servi par vLLM.

## 7. Alignement DPO

Le DPO part du **modèle SFT** et lui apprend à préférer la réponse `chosen` à la réponse `rejected` de chaque paire.

- **Modèle de départ** : modèle de base + adaptateur SFT **fusionné** (les poids LoRA sont ajoutés aux poids du modèle). TRL calcule les log-probabilités directement depuis `lm_head`, ce qui impose de fusionner l'adaptateur SFT avant d'en ajouter un nouveau.
- **Nouvel adaptateur LoRA** entraîné par DPO. La **référence** du DPO est le modèle SFT lui-même : TRL l'obtient en désactivant l'adaptateur, sans charger un second modèle en mémoire.
- **Données (v4)**, orientées triage et distinctes de celles du SFT, mélangées pour l'entraînement, évaluées séparément par langue :
  - **français** (MediQAl `mcqu` et `mcqm`) : `chosen` = corrigé, `rejected` = corrigé avec une erreur construite (omission, ajout ou substitution d'une proposition), sur des cas cliniques non retenus pour le SFT ;
  - **anglais** (UltraMedical) : préférences vérifiées par le corrigé (QCM) ou par un écart de note net (ChatDoctor), équilibrées sur la longueur ; test = paires revues par des experts (Medical RewardBench).
- **Reprise après redémarrage** : exécuter d'abord la section 1 (configuration), puis cette section. Le tokenizer et l'adaptateur SFT sont relus depuis `SFT_ADAPTER_DIR`. Avant la validation (7.4), exécuter aussi la cellule des fonctions de génération (section 6).
- **Relancer seulement la validation (7.4)**, sans réentraîner : section 1, puis les deux premières cellules de la section 7 (hyperparamètres, données 7.1), la cellule des fonctions de génération (section 6), et enfin la section 7.4. Sa première cellule recharge l'adaptateur DPO sauvegardé.

In [46]:
# Paires DPO de la phase 1 (v4, anonymisées), par langue
DPO_DATASETS = {"fr": "dpo_mediqal", "en": "dpo_ultramedical"}
DPO_COLUMNS = ["source", "lang", "pair_check", "triage", "prompt", "chosen", "rejected"]  # colonnes communes aux jeux

# Hyperparamètres DPO
DPO_BETA = 0.1              # écart toléré avec la référence : plus beta est grand, plus le modèle reste proche du SFT
DPO_LEARNING_RATE = 5e-5    # LoRA : ~1 % des poids entraînés ; à 5e-6 (valeur usuelle en DPO complet), les marges restaient quasi nulles
DPO_MAX_LENGTH = 1024       # prompt + réponse, en tokens
DPO_N_EVAL = 20 if SMALL_RUN else 200  # questions du test SFT français, par source, pour la génération (7.4)
DPO_BATCH_SIZE = 16                    # génération par lots : à baisser en cas d'OutOfMemoryError

SFT_ADAPTER_DIR = Path("../outputs") / RUN_NAME / "final"  # adaptateur SFT de la section 5
DPO_RUN_NAME = f"dpo-lora-r16-{DATA_VERSION}-beta{DPO_BETA}-lr{DPO_LEARNING_RATE}-seed{SEED}" + ("-small" if SMALL_RUN else "")
DPO_OUTPUT_DIR = Path("../outputs") / DPO_RUN_NAME  # checkpoints, adaptateur DPO et modèle final fusionné
print(f"SFT : {SFT_ADAPTER_DIR}\nDPO : {DPO_OUTPUT_DIR}")

SFT : ../outputs/sft-lora-r16-v4-seed42-small/final
DPO : ../outputs/dpo-lora-r16-v4-beta0.1-lr5e-05-seed42-small


### 7.1 Données DPO

In [47]:
# Tokenizer sauvegardé avec l'adaptateur SFT : même gabarit qu'à l'entraînement SFT
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR)


def pair_length(example):
    # Longueur en tokens de la plus longue des deux conversations (prompt + chosen, prompt + rejected)
    return max(
        len(tokenizer(tokenizer.apply_chat_template(example["prompt"] + answer, tokenize=False), add_special_tokens=False)["input_ids"])
        for answer in (example["chosen"], example["rejected"])
    )


DPO_SMALL_SIZES = {"train": 8, "validation": 2, "test": 4}  # petit run : paires par groupe
DPO_SMALL_LONGEST = 4                                       # + paires les plus longues du train, par langue

dpo_by_lang = {}
for lang, stem in DPO_DATASETS.items():
    files = {split: str(SFT_DATA_DIR / f"{stem}_{split}.anonymized.jsonl") for split in ("train", "validation", "test")}
    ds = load_dataset("json", data_files=files)
    # Paires trop longues : TRL couperait la fin de la réponse, on les écarte
    ds = ds.filter(lambda example: pair_length(example) <= DPO_MAX_LENGTH)
    if SMALL_RUN:
        # Quelques paires de chaque groupe évalué en 7.4 (fr : source × type d'erreur ; en : triage × chosen plus longue
        # ou plus courte), plus les paires les plus longues du train : mémoire GPU testée dans le pire cas
        for split, size in DPO_SMALL_SIZES.items():
            d = ds[split]
            groups = (list(zip(d["source"], d["rejected_type"])) if lang == "fr" else
                      [(t, len(c[0]["content"]) > len(r[0]["content"])) for t, c, r in zip(d["triage"], d["chosen"], d["rejected"])])
            keep = [i for group in sorted(set(groups)) for i in [j for j, g in enumerate(groups) if g == group][:size]]
            if split == "train":
                length = [pair_length(example) for example in d]
                keep += sorted(range(len(d)), key=lambda i: length[i], reverse=True)[:DPO_SMALL_LONGEST]
            ds[split] = d.select(sorted(set(keep)))
    if lang == "fr":  # type d'erreur de la rejected (omission, ajout, substitution), pour la validation 7.4
        fr_rejected_types = {split: d["rejected_type"] for split, d in ds.items()}
    dpo_by_lang[lang] = ds.select_columns(DPO_COLUMNS)
    print(f"{lang}: " + ", ".join(f"{split} {len(d)}" for split, d in ds.items()))

# Entraînement et validation : les langues mélangées
dpo_train = concatenate_datasets([ds["train"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
dpo_validation = concatenate_datasets([ds["validation"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
print(f"DPO : train {len(dpo_train)}, validation {len(dpo_validation)}")

fr: train 36, validation 8, test 16
en: train 36, validation 8, test 16
DPO : train 72, validation 16


In [48]:
# Une paire telle que la verra le modèle
example = dpo_train[0]
print(tokenizer.apply_chat_template(example["prompt"], add_generation_prompt=True, tokenize=False))
print(f"--- CHOSEN :\n{example['chosen'][0]['content']}\n--- REJECTED :\n{example['rejected'][0]['content']}")

<|im_start|>system
You are an expert medical assistant. Answer accurately and clearly.<|im_end|>
<|im_start|>user
A 25-year-old woman presents to the emergency department for the evaluation of a severe abdominal pain of 5 hours duration. The pain is colicky but is not localized. She also complains of nausea and an episode of vomiting. For the past 2 days, she has been constipated. She has had similar episodes of varying intensity in the past that resolved over a few hours. Several laboratory tests and imaging studies have been conducted in the past which were all within normal limits. The medical history is otherwise unremarkable. She denies smoking cigarettes or drinking alcohol. The vital signs are as follows: pulse 100/min, respiratory rate 16/min, and blood pressure 138/84 mm Hg. The physical examination reveals a young woman in obvious distress. There is no tenderness on abdominal examination. Laboratory tests are ordered, analgesics are administered, and the patient was admitted 

### 7.2 Modèle de départ : modèle SFT fusionné

In [49]:
# Libère la mémoire GPU du SFT si la section 5 vient de tourner
if "trainer" in globals():
    del trainer, model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Modèle de base + adaptateur SFT fusionné : point de départ et référence du DPO
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
print(f"{sft_model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2219.70it/s]


1.72 B paramètres


### 7.3 Entraînement DPO

- **LoRA** : même configuration que pour le SFT, sur un nouvel adaptateur.
- **Une époque** : le DPO sur-apprend vite. Au-delà, le modèle mémorise les paires sans mieux généraliser.
- **Batch effectif de 16** (2 × 8) : chaque exemple contient deux réponses, donc deux fois plus de tokens qu'en SFT.
- **Learning rate de 5e-5**, avec montée progressive sur 10 % des étapes. Plus élevé qu'en DPO complet (5e-6 à 5e-7) : avec LoRA, seule une petite partie des poids est entraînée. Au premier run, 5e-6 laissait la marge des récompenses quasi nulle sur le français et la courbe n'avait pas atteint son plateau. Si `logps/chosen` chute fortement, revenir à 2e-5.

In [50]:
dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# Suivi MLflow : même serveur que le SFT (connexion faite en section 5), expérience dédiée au DPO
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("qwen3-1.7b-dpo-lora")
mlflow.start_run(run_name=DPO_RUN_NAME)
mlflow.log_params({
    "model": MODEL_NAME, "data_version": DATA_VERSION, "sft_adapter": str(SFT_ADAPTER_DIR), "small_run": SMALL_RUN, "device": device,
    "beta": DPO_BETA, "lora_r": dpo_lora_config.r, "lora_alpha": dpo_lora_config.lora_alpha,
    **{f"pairs_{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
})

dpo_args = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    run_name=DPO_RUN_NAME,
    seed=SEED,
    # DPO
    beta=DPO_BETA,
    loss_type=["sigmoid"],  # perte DPO standard
    max_length=DPO_MAX_LENGTH,
    # Optimisation
    num_train_epochs=1,
    max_steps=10 if SMALL_RUN else -1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # batch effectif = 16 paires
    learning_rate=DPO_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,  # 10 % des étapes en montée progressive du learning rate
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=5 if SMALL_RUN else 25,
    save_steps=5 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=1 if SMALL_RUN else 5,
    report_to="mlflow",
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_validation,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)
dpo_trainer.model.print_trainable_parameters()

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Tokenizing train dataset: 100%|██████████| 72/72 [00:00<00:00, 232.82 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 72/72 [00:00<00:00, 10095.94 examples/s]
Tokenizing eval dataset: 100%|██████████| 16/16 [00:00<00:00, 182.73 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 16/16 [00:00<00:00, 2724.24 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [51]:
# Reprise après interruption : dpo_trainer.train(resume_from_checkpoint=True)
dpo_result = dpo_trainer.train()
dpo_result.metrics

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
5,0.685730,0.679909,0.710646,86753.000000,3.761461,4.471126,0.779392,-0.015294,-0.049091,0.562500,0.033797,-209.801632,-188.913902
10,0.526834,0.676756,0.716662,173506.000000,3.812221,4.509657,0.779211,-0.027093,-0.074214,0.687500,0.047121,-209.919624,-189.165123


{'train_runtime': 114.2104,
 'train_samples_per_second': 1.401,
 'train_steps_per_second': 0.088,
 'total_flos': 1841894021210112.0,
 'train_loss': 0.6274661481380462,
 'epoch': 2.0}

In [52]:
# Sauvegarde de l'adaptateur DPO (meilleur checkpoint) et de la configuration du run
DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_trainer.save_model(str(DPO_FINAL_DIR))
tokenizer.save_pretrained(str(DPO_FINAL_DIR))

dpo_run_config = {
    "model": MODEL_NAME,
    "sft_adapter": str(SFT_ADAPTER_DIR),  # l'adaptateur DPO s'applique au modèle SFT fusionné
    "seed": SEED,
    "small_run": SMALL_RUN,
    "beta": DPO_BETA,
    "lora": {"r": dpo_lora_config.r, "alpha": dpo_lora_config.lora_alpha, "dropout": dpo_lora_config.lora_dropout},
    "data": {f"{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
    "training_args": dpo_args.to_dict(),
}
(DPO_FINAL_DIR / "run_config.json").write_text(json.dumps(dpo_run_config, indent=2, default=str), encoding="utf-8")

4902

### 7.4 Validation DPO

1. **Préférences** sur la validation et le test de chaque langue (test anglais : revu par des experts), et sur trois découpages du test :
   - **anglais, selon la longueur** (`chosen` plus longue ou plus courte que la `rejected`) : une précision très différente entre les deux révèle une préférence apprise pour la longueur plutôt que pour le fond ;
   - **anglais, triage** : paires dont la question décrit un patient, avec constantes ou vocabulaire d'urgence (`triage`), le plus proche du cas d'usage ;
   - **français, selon le type d'erreur de la `rejected`** : *omission* (une bonne proposition retirée), *ajout* (une mauvaise ajoutée), *substitution* (l'une remplacée par l'autre). Un modèle qui a appris « cocher moins » plutôt que « cocher juste » réussit les ajouts et rate les omissions.

   La *récompense implicite* d'une réponse mesure combien le modèle DPO la juge plus probable que le modèle SFT. On en tire :
   - `reward_accuracy` : part des paires où la récompense de la `chosen` dépasse celle de la `rejected` (0,5 = hasard) ;
   - `reward_margin` : écart moyen entre les deux récompenses ;
   - `logps_chosen` / `logps_rejected` : log-probabilités moyennes des deux réponses sous le modèle DPO.
2. **Réponses générées** sur le test SFT français (QCM `mcqu` et `mcqm`, chaque question avec l'historique de son cas) : modèle SFT vs modèle DPO, avec les mesures et les réponses naïves de la section 6, et le décompte des questions où un seul des deux modèles a la bonne réponse. On mesure l'effet du DPO sur les décisions, au-delà des préférences.


In [53]:
# Modèle DPO pour la validation : celui de 7.3 s'il est encore en mémoire, sinon l'adaptateur sauvegardé.
# Après 7.5, l'adaptateur DPO est fusionné dans les poids : désactiver l'adaptateur ne donnerait plus le modèle SFT
# (référence du DPO) et toutes les récompenses seraient nulles. Il faut alors recharger.
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_in_memory = "dpo_trainer" in globals() and any("lora_" in name for name, _ in dpo_trainer.model.named_parameters())
if not dpo_in_memory:
    for name in ("dpo_trainer", "dpo_model", "merged_model", "sft_model", "base_model", "trainer", "model"):
        globals().pop(name, None)
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    # Comme en 7.2 et 7.3 : modèle SFT fusionné + nouvel adaptateur ; la référence reste le modèle SFT (adaptateur désactivé).
    # Ne pas passer un PeftModel déjà chargé : TRL prendrait une copie de l'adaptateur DPO comme référence.
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
    sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
    eval_args = DPOConfig(
        output_dir=str(DPO_OUTPUT_DIR / "eval"), seed=SEED, beta=DPO_BETA, loss_type=["sigmoid"], max_length=DPO_MAX_LENGTH,
        per_device_eval_batch_size=4, bf16=bf16, fp16=dtype == torch.float16, report_to="none",
    )
    dpo_trainer = DPOTrainer(
        model=sft_model, args=eval_args, train_dataset=dpo_validation.select(range(1)),  # train_dataset exigé par TRL, inutilisé
        processing_class=tokenizer, peft_config=LoraConfig.from_pretrained(str(DPO_FINAL_DIR)),
    )
    # Poids de l'adaptateur DPO sauvegardé en 7.3, dans le nouvel adaptateur
    result = set_peft_model_state_dict(dpo_trainer.model, load_file(str(DPO_FINAL_DIR / "adapter_model.safetensors")))
    assert not result.unexpected_keys, result.unexpected_keys[:5]
    assert not [key for key in result.missing_keys if "lora_" in key], "poids LoRA manquants"
print(f"Modèle DPO : {'en mémoire (7.3)' if dpo_in_memory else f'rechargé depuis {DPO_FINAL_DIR}'}")

Modèle DPO : en mémoire (7.3)


In [54]:
# 1. Préférences, par split et par langue
# (TRL nomme ses métriques « eval_rewards/... » quel que soit le préfixe : on les renomme ici)
eval_sets = {f"{split}_{lang}": ds[split] for lang, ds in dpo_by_lang.items() for split in ("validation", "test")}

# Test anglais selon la longueur : la chosen est-elle plus longue ou plus courte que la rejected ?
test_en = dpo_by_lang["en"]["test"]
chosen_longer = [len(c[0]["content"]) > len(r[0]["content"]) for c, r in zip(test_en["chosen"], test_en["rejected"])]
eval_sets["test_en_chosen_longer"] = test_en.select([i for i, longer in enumerate(chosen_longer) if longer])
eval_sets["test_en_chosen_shorter"] = test_en.select([i for i, longer in enumerate(chosen_longer) if not longer])

# Test anglais, paires proches du triage (patient décrit, constantes ou vocabulaire d'urgence)
eval_sets["test_en_triage"] = test_en.select([i for i, triage in enumerate(test_en["triage"]) if triage])

# Test français selon le type d'erreur de la rejected
for rejected_type in ("omission", "ajout", "substitution"):
    indices = [i for i, t in enumerate(fr_rejected_types["test"]) if t == rejected_type]
    eval_sets[f"test_fr_{rejected_type}"] = dpo_by_lang["fr"]["test"].select(indices)

dpo_rows = {}
for name, dataset in eval_sets.items():
    if len(dataset) == 0:  # petit run : un sous-ensemble peut être vide
        continue
    m = dpo_trainer.evaluate(dataset, metric_key_prefix=name)
    dpo_rows[name] = {
        "pairs": len(dataset),
        "loss": m[f"{name}_loss"],
        "reward_accuracy": m["eval_rewards/accuracies"],
        "reward_margin": m["eval_rewards/margins"],
        "logps_chosen": m["eval_logps/chosen"],
        "logps_rejected": m["eval_logps/rejected"],
    }
dpo_summary = pd.DataFrame(dpo_rows).T
dpo_summary

Tokenizing eval dataset: 100%|██████████| 8/8 [00:00<00:00, 153.08 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 8/8 [00:00<00:00, 1365.39 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.526834,No log,10,0.691762,0.147311,173506.000000,5.255216,5.204350,0.957399,-0.002191,-0.005106,0.625000,0.002915,-3.602860,-3.703690


Tokenizing eval dataset: 100%|██████████| 16/16 [00:00<00:00, 148.50 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 16/16 [00:00<00:00, 2680.71 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.526834,No log,10,0.690414,0.126723,173506.000000,5.891692,6.057999,0.963941,0.003016,-0.002528,0.625000,0.005544,-3.613104,-3.712861


Tokenizing eval dataset: 100%|██████████| 8/8 [00:00<00:00, 138.95 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 8/8 [00:00<00:00, 1813.85 examples/s]


Training Loss,Validation Loss,Step,En Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.526834,No log,10,0.672388,0.801389,173506.000000,3.937054,4.091211,0.755045,-0.117100,-0.174437,0.750000,0.057336,-416.567490,-374.847229


... (11 outputs left out)

Training Loss,Validation Loss,Step,Fr Ajout Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.526834,No log,10,0.701076,0.142449,173506.000000,7.447870,7.184779,0.950355,-0.013958,0.001759,0.250000,-0.015716,-4.176192,-4.029274


Tokenizing eval dataset: 100%|██████████| 8/8 [00:00<00:00, 192.41 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 8/8 [00:00<00:00, 1749.45 examples/s]


Training Loss,Validation Loss,Step,Fr Substitution Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.526834,No log,10,0.692920,0.125888,173506.000000,5.918015,5.865613,0.967328,-0.002455,-0.002963,0.500000,0.000508,-2.680658,-3.118386


,pairs,loss,reward_accuracy,reward_margin,logps_chosen,logps_rejected
validation_fr,8.0,0.691762,0.6250,0.002915,-3.602860,-3.703690
test_fr,16.0,0.690414,0.6250,0.005544,-3.613104,-3.712861
validation_en,8.0,0.672388,0.7500,0.057336,-416.567490,-374.847229
test_en,16.0,0.681074,0.6875,0.041510,-453.246895,-411.893898
test_en_chosen_longer,8.0,0.677998,0.7500,0.060515,-474.498932,-336.884781
test_en_chosen_shorter,8.0,0.647491,0.7500,0.104371,-431.782715,-487.261856
test_en_triage,8.0,0.695991,0.5000,0.007149,-463.755936,-411.790817
test_fr_omission,4.0,0.692239,0.2500,0.001855,-5.078286,-4.579793
test_fr_ajout,4.0,0.701076,0.2500,-0.015716,-4.176192,-4.029274
test_fr_substitution,8.0,0.692920,0.5000,0.000508,-2.680658,-3.118386


In [55]:
# 2. Réponses générées sur le test SFT français (QCM) : modèle SFT (adaptateur DPO désactivé) vs modèle DPO.
# Mêmes questions qu’en section 6 : effet du DPO sur les décisions, au-delà des préférences.
assert "generate_batch" in globals(), "Exécuter d'abord la cellule des fonctions de génération (section 6)"
DPO_GENERATION_SOURCES = {"sft_mediqal_mcqu": MAX_NEW_TOKENS, "sft_mediqal_mcqm": 192}  # réponses mcqm : listes de 2 à 5 options
DPO_ANSWER_COLUMNS = ["sft", "dpo", *NAIVE_ANSWERS]

dpo_model = dpo_trainer.model.eval()
dpo_model.config.use_cache = True  # accélère la génération

frames = []
for source, max_new_tokens in DPO_GENERATION_SOURCES.items():
    sft_data = load_dataset("json", data_files={s: str(SFT_DATA_DIR / f"{source}_{s}.anonymized.jsonl") for s in ("train", "test")})
    prompts, references = question_examples(sft_data["test"])  # chaque question avec l'historique de son cas
    prompts, references = prompts[:DPO_N_EVAL], references[:DPO_N_EVAL]
    with dpo_model.disable_adapter():
        sft_answers = generate_batch(prompts, DPO_BATCH_SIZE, max_new_tokens, desc=f"SFT {source}", model=dpo_model)
    dpo_answers = generate_batch(prompts, DPO_BATCH_SIZE, max_new_tokens, desc=f"DPO {source}", model=dpo_model)
    frames.append(pd.DataFrame({
        "source": source,
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": references,
        "sft": sft_answers,
        "dpo": dpo_answers,
        **naive_answers(question_examples(sft_data["train"])[1]),  # mêmes réponses pour toutes les questions de la source
    }))
dpo_generations = add_scores(pd.concat(frames, ignore_index=True), DPO_ANSWER_COLUMNS)
dpo_generations.to_csv(DPO_FINAL_DIR / "test_fr_generations.csv", index=False)

# Questions où un seul des deux modèles a exactement la bonne réponse (test de McNemar : ce sont elles qui comptent)
only_dpo = int(((dpo_generations["accuracy_dpo"] == 1) & (dpo_generations["accuracy_sft"] == 0)).sum())
only_sft = int(((dpo_generations["accuracy_sft"] == 1) & (dpo_generations["accuracy_dpo"] == 0)).sum())
print(f"Bonne réponse du DPO seul : {only_dpo} questions ; du SFT seul : {only_sft} questions")

display(pd.concat({source: score_table(group, DPO_ANSWER_COLUMNS) for source, group in dpo_generations.groupby("source")}))
dpo_comparison = score_table(dpo_generations, DPO_ANSWER_COLUMNS)
dpo_comparison


DPO sft_mediqal_mcqm: 100%|██████████| 2/2 [01:03<00:00, 31.75s/lot]

Bonne réponse du DPO seul : 0 questions ; du SFT seul : 2 questions


sft      dpo  frequente  tout_cocher
sft_mediqal_mcqm accuracy         0.050000  0.05000   0.000000     0.000000
                 option_accuracy  0.580000  0.59000   0.530000     0.490000
                 f1               0.599762  0.60869   0.571905     0.649802
                 extra_letters    0.400000  0.45000   0.550000     2.550000
sft_mediqal_mcqu accuracy         0.400000  0.30000   0.200000     0.000000
                 option_accuracy  0.760000  0.72000   0.680000     0.200000
                 f1               0.400000  0.30000   0.200000     0.333333
                 extra_letters    0.000000  0.00000   0.000000     4.000000

,sft,dpo,frequente,tout_cocher
accuracy,0.225000,0.175000,0.100000,0.000000
option_accuracy,0.670000,0.655000,0.605000,0.345000
f1,0.499881,0.454345,0.385952,0.491567
extra_letters,0.200000,0.225000,0.275000,3.275000


In [56]:
# Quelques exemples à relire
dpo_generations.groupby("source")[["source", "question", "reference", "sft", "dpo"]].head(3)

,source,question,reference,sft,dpo
0,sft_mediqal_mcqu,entation étiologique :\n\nPropositions :\nA. Bilatéralité des lésions\nB. Femme jeune\nC. Association à des arthralgies\nD. Association à une fièvre\nE. Episode diarrhéique\n\nUne seule propositio...,E. Episode diarrhéique,C. Association à des arthralgies,C. Association à des arthralgies
1,sft_mediqal_mcqu,r le diagnostic ?\n\nPropositions :\nA. Hémoculture\nB. Lavement baryté\nC. Colonoscopie\nD. Sérodiagnostic de Yersinia enterocolitica\nE. Examen parasitologique des selles\n\nUne seule propositio...,D. Sérodiagnostic de Yersinia enterocolitica,B. Lavement baryté,C. Colonoscopie
2,sft_mediqal_mcqu,Uvéite\nB. Adénopathies hilobronchiques bilatérales\nC. Paralysie faciale périphérique\nD. Sérologie de mycoplasme positive\nE. Adénopathies épitrochléennes bilatérales\n\nUne seule proposition e...,B. Adénopathies hilobronchiques bilatérales,B. Adénopathies hilobronchiques bilatérales,B. Adénopathies hilobronchiques bilatérales
20,sft_mediqal_mcqm,. Myélogramme\nC. Gastroscopie avec biopsie\nD. Dosages de vitamine B12 et de folates dans le sang\nE. Recherche d'anticorps anti-facteur intrinsèque\n\nPlusieurs propositions sont exactes (au moi...,- B. Myélogramme\n- D. Dosages de vitamine B12 et de folates dans le sang,A. Tubage gastrique\nB. Myélogramme\nC. Gastroscopie avec biopsie\nD. Dosages de vitamine B12 et de folates dans le sang,A. Tubage gastrique\nB. Myélogramme\nC. Gastroscopie avec biopsie\nD. Dosages de vitamine B12 et de folates dans le sang
21,sft_mediqal_mcqm,B. Trouble d'absorption de la vitamine B12\nC. Carence d'apport en folates\nD. Trouble d'absorption des folates\nE. Trouble d'utilisation des folates\n\nPlusieurs propositions sont exactes (au moi...,- C. Carence d'apport en folates\n- D. Trouble d'absorption des folates,- A. Carence d'apport en vitamine B12\n- D. Trouble d'absorption des folates,- A. Carence d'apport en vitamine B12\n- D. Trouble d'absorption des folates
22,sft_mediqal_mcqm,phénotypes rhésus et Kell\nC. Recherche et identification des anticorps sériques\nD. Test de Coombs direct\nE. Recherche d'anticorps anti-plaquettes\n\nPlusieurs propositions sont exactes (au moi...,- B. Détermination des phénotypes rhésus et Kell\n- C. Recherche et identification des anticorps sériques\n- D. Test de Coombs direct,A. Recherche d'anticorps anti-HLA\nB. Détermination des phénotypes rhésus et Kell\nC. Recherche et identification des anticorps sériques\nD. Test de Coombs direct,A. Recherche d'anticorps anti-HLA\nB. Détermination des phénotypes rhésus et Kell\nC. Recherche et identification des anticorps sériques\nD. Test de Coombs direct


In [57]:
# Métriques finales dans MLflow, puis clôture du run
# Validation relancée seule : les métriques rejoignent le run DPO existant (même nom), au lieu d'en créer un nouveau
if mlflow.active_run() is None:
    if MLFLOW_BACKEND == "dagshub":
        import dagshub

        dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
    else:
        mlflow.set_tracking_uri("sqlite:///../mlflow.db")
    previous = mlflow.search_runs(
        experiment_names=["qwen3-1.7b-dpo-lora"], filter_string=f"tags.mlflow.runName = '{DPO_RUN_NAME}'",
        order_by=["start_time DESC"], max_results=1,
    )
    assert len(previous), f"run {DPO_RUN_NAME} introuvable dans MLflow"
    mlflow.start_run(run_id=previous["run_id"].iloc[0])
mlflow.log_metrics({
    f"{name}_{metric}": value
    for name, row in dpo_summary.iterrows()
    for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_fr_{metric}_{model}": dpo_comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in dpo_comparison.columns
})
mlflow.log_metrics({"test_fr_only_dpo_correct": only_dpo, "test_fr_only_sft_correct": only_sft})
mlflow.end_run()

🏃 View run dpo-lora-r16-v4-beta0.1-lr5e-05-seed42-small at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3/runs/8dd2d75da6a44f5083495e6e0faee3d6
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3


### 7.5 Modèle final fusionné

Modèle de base + SFT + DPO en un seul jeu de poids complets (environ 3,4 Go en bf16), avec le tokenizer et son gabarit : c'est le modèle à servir avec vLLM en phase 3.

In [58]:
MERGED_DIR = DPO_OUTPUT_DIR / "merged"
merged_model = dpo_trainer.model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Modèle final : {MERGED_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.66s/it]

Modèle final : ../outputs/dpo-lora-r16-v4-beta0.1-lr5e-05-seed42-small/merged


## Lecture des résultats DPO

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | perte DPO d'entraînement et de validation (0,69 = aucune préférence apprise) | baissent sous 0,69 ; si `eval_loss` remonte : surapprentissage |
| `rewards/accuracies` | part des paires où la `chosen` l'emporte sur la `rejected` | monte nettement au-dessus de 0,5 |
| `rewards/margins` | écart moyen de récompense entre `chosen` et `rejected` | augmente, sans exploser |
| `logps/chosen`, `logps/rejected` | log-probabilités moyennes des deux réponses | `rejected` baisse plus que `chosen` ; si `chosen` chute fortement, baisser le learning rate ou augmenter beta |
| `{validation,test}_{fr,en}_*` | les mêmes métriques, par split et par langue (test anglais : revu par des experts) | proches entre validation et test |
| `test_en_chosen_{longer,shorter}_*` | test anglais, selon que la `chosen` est la plus longue ou la plus courte | précisions proches ; un grand écart = préférence pour la longueur |
| `test_en_triage_*` | test anglais, paires proches du triage (patient décrit, constantes ou urgence) | au moins aussi bon que le test entier |
| `test_fr_{omission,ajout,substitution}_*` | test français, selon le type d'erreur de la `rejected` | précisions proches ; ajout ≫ omission = le modèle a appris à cocher moins |
| `test_fr_{accuracy,option_accuracy,f1,extra_letters}_*` | lettres des réponses générées sur le test SFT français (`mcqu`, `mcqm`), comparées au corrigé (mêmes mesures et réponses naïves qu'en section 6) | dpo ≥ sft, au-dessus de `frequente` et `tout_cocher` |
| `test_fr_only_{dpo,sft}_correct` | questions où un seul des deux modèles a exactement la bonne réponse | only_dpo > only_sft |

- Les courbes `eval_rewards/*` mélangent toutes les évaluations (TRL ignore le préfixe) : pour la comparaison finale, utiliser `{validation,test}_{fr,en}_*`.
- **Itérer** : un run par jeu d'hyperparamètres (`DPO_BETA`, `DPO_LEARNING_RATE`), avec le même `SEED`. Le nom du run les contient.